#  天猫订单数据可视化

### 一、数据理解
本数据集共收集了天猫发生在2020年2月内的28010条数据。

共7个字段说明

1.订单编号：订单编号

2.总金额：订单总金额

3.买家实际支付金额：总金额 - 退款金额（在已付款的情况下）。金额为0（在未付款的情况下）

4.收货地址：各个省份

5.订单创建时间：下单时间

6.订单付款时间：付款时间

7.退款金额：付款后申请退款的金额。如无付过款，退款金额为0


### 二、分析目的
1.订单数在地图上的分布

2.每日订单数量分析（按照订单创建时间分组，得到每一天的订单量）

3.每小时订单数量分析

4.订单每个环节的转化转化率

### 三、数据分析与可视化过程

#### 1、导入需要的库、编码、路径设置

#### 2、导入数据并进行预处理

订单付款时间存在缺失，预计是未付款订单，不做处理

In [ ]:
import pandas as pd
df = pd.read_csv('data/tmall_order_report.csv')
df.head()
df

In [ ]:
df.info()

#### 重复值

In [ ]:
df.duplicated().sum()


#### 缺失值

In [ ]:
df.isnull().sum()   # 订单付款时间 有2923个缺失值，属于正常现象，说明这些单位付过款，无需处理


#### 去掉属性列名的空格

In [ ]:
df.columns


In [ ]:
#发现有字段中有 空格
df.rename(columns=(lambda i:i.strip()),inplace=True)

### 3、各省份的订单量地图可视化

In [ ]:
df.收货地址.value_counts()

In [ ]:
df.订单付款时间.notnull()  #查着订单付款时间不为空的
df[df.订单付款时间.notnull()]   #筛选没有付款的订单数据

In [ ]:
def province_map(p):
    if p in ['北京','天津','上海','重庆']:
        return p +'市'
    return p
df.收货地址 = df.收货地址.map(province_map)

In [ ]:
df.收货地址.value_counts()

In [ ]:
result = df[df.订单付款时间.notnull()].groupby('收货地址')[['订单编号']].count()  #统计各个省的订单量
result

In [ ]:
result2 = result.to_dict()['订单编号']
result2

In [ ]:
list(result2.items())

In [ ]:
import pyecharts.options as opts
from pyecharts.charts import Map
map1 = (
    Map()
    .add(
         "各省份订单数",
        list(result2.items()),
        'china',
        is_map_symbol_show = False
    )
    .set_global_opts(title_opts = opts.TitleOpts(title = "各省份订单量"),
        visualmap_opts=opts.VisualMapOpts(max_ = 3060)
    )   
    
    
)
map1.load_javascript()


In [ ]:
map1.render_notebook()

分析：从图中可以看出，上海、广东、北京、江浙及四川的订单量普遍较高。

### 4、每日订单数量分析（时间序列分析）
#### 按照订单创建时间分组，得到每一天的订单量 

时间列格式为object，需要修改为datetime

In [ ]:
df['订单创建时间']=pd.to_datetime(df['订单创建时间'])
df['订单付款时间']=pd.to_datetime(df['订单付款时间'])
df[df.订单付款时间.notnull()]

In [ ]:
df.订单创建时间

In [ ]:
order_add_time = df.订单创建时间.map(lambda x: x.strftime('%Y-%m-%d'))

In [ ]:
result3 = df.groupby(order_add_time)[['订单编号']].count().to_dict()['订单编号']  #按照订单创建时间分组，得到每一天的订单量
result3

In [ ]:
line = (
    Line()
    .add_xaxis(list(result3.keys()))
    .add_yaxis(
        '订单量',
        list(result3.values())
       )
    .set_global_opts( 
        title_opts = opts.TitleOpts(title = "每天订单量"),
        yaxis_opts = opts.AxisOpts(
              splitline_opts = opts.SplitLineOpts(is_show = True)
        
        )
    )
    #标签配置项
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show= False ),
        markpoint_opts= opts.MarkPointOpts(
            data=[
                opts.MarkPointItem(type_='max')
            ]
        )
    )
                     
      
)
line. render_notebook()

#### 每日的订单量最高在2月25日，达到3416，2月10至2月14日，订单量很低，几乎为0。

### 5、每小时订单量统计可视化

In [ ]:
order_add_time2 = df.订单创建时间.map(lambda x: x.strftime('%H'))
order_add_time2.value_counts()

In [ ]:
result4 = df.groupby(order_add_time2)[['订单编号']].count().to_dict()['订单编号']
result4

In [ ]:
bar = (
    Bar()
    .add_xaxis(list(result4.keys()))
    .add_yaxis(
        '订单量',
        list(result4.values())
       )
    .set_global_opts( 
        title_opts = opts.TitleOpts(title = "每小时订单量"),
        yaxis_opts = opts.AxisOpts(
              splitline_opts = opts.SplitLineOpts(is_show = True)
        
        )
    )
    #标签配置项
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show= False ),
        markpoint_opts= opts.MarkPointOpts(
            data=[
                opts.MarkPointItem(type_='max')
            ]
        )
    )
                     
      
)
bar.render_notebook()

#### 分析：根据每小时订单量柱状图可以得出：凌晨3~5点，下单的人数较少，晚上9点的订单量最多。

### 6、订单转化率分析
根据上述信息，无法确定分析方向，所以假设2月总销售目标是220万，为了进一步了解数据，可以从不同环节转化率来看看是哪个环节出了问题。

用户行为转化路径：创建--付款--实付--全额

这里采用绝对转化率实现：1.求出各环节的订单数；2.求转化率：用本环节的订单数除以订单创建数


In [ ]:
# 看各个环节的漏斗------创建--付款--实际--全额 每一层的转化是多少
rates=pd.Series({'下单':df['订单创建时间'].count(),
'付款':df['订单付款时间'].count(),
'实付款':df[df['买家实际支付金额']>0].shape[0],  #巧妙运算  df[df['买家实际支付金额']>0].shape(0)，shape(0) 将返回数据的行数。
'全额付款':df[df['买家实际支付金额']==df['总金额']].shape[0]},name='订单量').to_frame()  #to_frame(): 将字典转换为DataFrame。
 
rates
 
rates['转化率']=rates['订单量'].apply(lambda x:round(x/rates.iloc[0,0],3)) #添加
rates['相对转化率']=round((rates/rates.shift())['订单量'].fillna(1),3)  #添加
 
print(rates)
 
#展示漏斗图
c=(fu().add('',[list(z) for z in zip(rates.index,rates['转化率'])],
            label_opts=opts.LabelOpts(position='inside',formatter='{b}:{c}'))).set_global_opts(title_opts=opts.TitleOpts(title='整体转化率（%）'))
c.render_notebook()

#### 分析：根据TrustData的报告显示，淘宝2015年平时的订单成功率为97.4%。而本次分析的付款转化率约86%低于预期标准，实际成交及全额成交的环节转化率不到70%。属于比较低的水平。

### 四、总结
1、由订单数在地图上的分布来看，上海、广东、北京、江浙及四川的订单量普遍较高。上海市为全国最高，达到3060。

2、每日的订单量最高在2月25日，达到3416，2月10至2月14日，订单量很低，几乎为0。而根据每小时订单量统计可以得出：凌晨3~5点，下单的人数较少，晚上9点的订单量最多。

3、转化率情况：付款成功率为86%，低于正常水平的97.4%，实际成交和全额成交的转化率分别为67.7%和65.8%。